# Preprocessing Validation (Sessions_In)

This notebook mirrors the `eda preprocess` CLI flow using a minimal test corpus from `Sessions_In`.

It does three things:
1. Builds a temporary vault-like structure with a `Sessions/` folder.
2. Loads markdown files and verifies frontmatter + description extraction.
3. Runs the preprocess CLI against that staged input.

In [6]:
from pathlib import Path
import shutil
import sys
import pandas as pd

# Resolve workspace root robustly regardless of notebook cwd.
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
workspace = next((p for p in _candidates if (p / "src" / "eda").exists()), Path.cwd())
eda_src = workspace / "src" / "eda"
if str(eda_src) not in sys.path:
    sys.path.insert(0, str(eda_src))

from eda.data.loader import load_vault
print(f"workspace={workspace}")
print(f"eda_src={eda_src}")

workspace=c:\projects\fab\Fabcon2026
eda_src=c:\projects\fab\Fabcon2026\src\eda


In [9]:
source_dir = workspace / "Sessions_In"
staged_vault = workspace / "Processed" / "sessions_in_vault"
staged_sessions = staged_vault / "Sessions"

staged_sessions.mkdir(parents=True, exist_ok=True)

for md_file in source_dir.glob("*.md"):
    shutil.copy2(md_file, staged_sessions / md_file.name)

print(f"Staged {len(list(staged_sessions.glob('*.md')))} markdown file(s) in: {staged_sessions}")

Staged 1 markdown file(s) in: c:\projects\fab\Fabcon2026\Processed\sessions_in_vault\Sessions


In [8]:
df = load_vault(staged_vault)

cols = [
    "file",
    "title",
    "day",
    "track",
    "session_type",
    "frontmatter_raw",
    "description",
    "markdown_blob",
]

preview = df[cols].copy()
preview["frontmatter_raw"] = preview["frontmatter_raw"].str.slice(0, 160)
preview["description"] = preview["description"].str.slice(0, 220)
preview["markdown_blob"] = preview["markdown_blob"].str.slice(0, 160)
preview

,file,title,day,track,session_type,frontmatter_raw,description,markdown_blob
0,"90 Days, 1 Fabric Foundation- Delivering SQL M...","90 Days, 1 Fabric Foundation: Delivering SQL M...",Friday,Data Integration,Sponsor Speaker 60 Minute Session (as part of ...,"title: ""90 Days, 1 Fabric Foundation: Deliveri...","In this case study, Protective Industries tran...","---\ntitle: ""90 Days, 1 Fabric Foundation: Del..."


In [12]:
import os
import subprocess

output_base = workspace / "Processed" / "sessions_in_preprocessed"

cmd = [
    sys.executable,
    "-m",
    "eda.main",
    "preprocess",
    "--vault",
    str(staged_vault),
    "--sessions-dir",
    "Sessions",
    "--no-workshops",
    "--no-tfidf",
    "--format",
    "csv",
    "--output",
    str(output_base),
]

env = os.environ.copy()
env["PYTHONPATH"] = str((workspace / "src" / "eda").resolve()) + os.pathsep + env.get("PYTHONPATH", "")
env["PYTHONIOENCODING"] = "utf-8"

result = subprocess.run(cmd, capture_output=True, text=False, env=env)
stdout = result.stdout.decode("utf-8", errors="replace")
stderr = result.stderr.decode("utf-8", errors="replace")
print(stdout)
if result.returncode != 0:
    print(stderr)
    raise RuntimeError(f"preprocess command failed with exit code {result.returncode}")

Loading sessions from: c:\projects\fab\Fabcon2026\Processed\sessions_in_vault
Loaded 1 sessions
         Dataset Overview         
┌────────────────────────┬───────┐
│ Metric                 │ Value │
├────────────────────────┼───────┤
│ Total sessions         │ 1     │
│   FABCON               │ 1     │
│   Friday               │ 1     │
│ Unique tracks          │ 1     │
│   Level 300            │ 1     │
│   Status: Not Reviewed │ 1     │
└────────────────────────┴───────┘
Saved → c:\projects\fab\Fabcon2026\Processed\sessions_in_preprocessed.csv



In [13]:
processed_csv = output_base.with_suffix(".csv")
processed_df = pd.read_csv(processed_csv)
processed_df[["file", "title", "day", "track", "description", "frontmatter_raw"]].head()

,file,title,day,track,description,frontmatter_raw
0,"90 Days, 1 Fabric Foundation- Delivering SQL M...","90 Days, 1 Fabric Foundation: Delivering SQL M...",Friday,Data Integration,"In this case study, Protective Industries tran...","title: ""90 Days, 1 Fabric Foundation: Deliveri..."
